# 05. 팽창 Mask Contrast Threshold 함수

실제 목표는 SegFormer featuremap 또는 probability map에서 출발하는 것이다. 현재는 그 값을 알 수 없으므로 GT mask를 랜덤 팽창한 candidate mask를 입력으로 사용한다.

이 노트북의 함수 입력은 `raw image + GT보다 넓은 candidate mask`이고, 출력은 scratch 대비 정도를 나타내는 scalar `threshold`이다.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "scratch_postprocess_utils.py").exists():
    matches = list(Path.cwd().glob("**/Scratch_Postprocess/scratch_postprocess_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
sys.path.insert(0, str(NOTEBOOK_DIR))

from scratch_postprocess_utils import *

ensure_dirs()
print("project:", NOTEBOOK_DIR)

## 알고리즘

넓은 mask 내부에는 scratch와 주변 background가 함께 들어간다. 평균 대비를 쓰면 scratch 신호가 희석되므로 내부 pixel contrast의 상위 quantile을 사용한다.

```text
background_rgb = median(raw pixels in outer ring)
mask_contrast_q = quantile(contrast(candidate mask pixels, background_rgb), 0.90)
background_contrast_q = quantile(contrast(ring pixels, background_rgb), 0.90)
threshold = mask_contrast_q
background_excess_threshold = max(mask_contrast_q - background_contrast_q, 0)
```

`threshold`는 raw contrast severity score이고, `background_excess_threshold`는 background texture를 보수적으로 뺀 보조 score이다. 둘 다 micro/scratch 최종 기준이 아니라 실제 데이터 calibration에 넘길 값이다.

## Trade-off

- 평균 contrast는 안정적이지만 팽창 mask에서 background pixel 때문에 희석된다.
- 상위 quantile contrast는 넓은 mask에 강하지만 texture peak나 specular noise에 민감할 수 있다.
- 가까운 ring은 local background를 잘 잡지만 scratch halo가 섞일 수 있다.
- 먼 ring은 contamination에 강하지만 shading 변화에 약하다.
- RGB Euclidean은 색상 scratch에 유리하고, luma는 단순하지만 색상 차이를 놓칠 수 있다.

## 05-1. 단일 샘플에서 랜덤 팽창 mask 만들기

In [ ]:
manifest_path = DATA_ROOT / "metadata" / "samples.csv"
if not manifest_path.exists():
    manifest = generate_random_scratch_dataset(n_samples=120, size=640, seed=7, overwrite=True)
else:
    manifest = pd.read_csv(manifest_path)

rng = set_seed(507)
sample = manifest.sample(1, random_state=5).iloc[0]
image = load_image(sample["image_path"])
gt_mask = load_mask(sample["mask_path"])
candidate_mask, dilation_radius = random_dilate_mask(gt_mask, rng, min_radius=4, max_radius=14)

result = estimate_scratch_contrast_threshold(
    image=image,
    candidate_mask=candidate_mask,
    metric="rgb_euclidean",
    mask_quantile=0.90,
    background_quantile=0.90,
    ring_gap=2,
    ring_radius=14,
)
result

## 05-2. 입력 mask와 출력 threshold 시각화

In [ ]:
component_table = estimate_component_contrast_thresholds(
    image=image,
    candidate_mask=candidate_mask,
    metric="rgb_euclidean",
    mask_quantile=0.90,
    background_quantile=0.90,
    ring_gap=2,
    ring_radius=14,
)
display(component_table)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image)
axes[0].set_title("raw image")
axes[0].axis("off")
axes[1].imshow(gt_mask, cmap="gray")
axes[1].set_title("GT mask")
axes[1].axis("off")
axes[2].imshow(candidate_mask, cmap="gray")
axes[2].set_title(f"dilated candidate mask r={dilation_radius}")
axes[2].axis("off")
fig.suptitle(
    f"threshold={result['threshold']:.2f}, excess={result['background_excess_threshold']:.2f}, z={result['contrast_z_excess']:.2f}, metric={result['metric']}"
)
plt.tight_layout()
plt.savefig(RUNS_ROOT / "dilated_mask_threshold_single_example.png", dpi=150)
plt.show()

## 05-3. 전체 데이터셋에서 랜덤 팽창 반복 평가

In [ ]:
eval_df = evaluate_random_dilated_thresholds(
    manifest,
    n_repeats=3,
    min_radius=2,
    max_radius=14,
    metric="rgb_euclidean",
    mask_quantile=0.90,
    background_quantile=0.90,
    seed=1705,
)
eval_path = RUNS_ROOT / "dilated_mask_threshold_eval.csv"
eval_df.to_csv(eval_path, index=False, encoding="utf-8-sig")
print(eval_path)
display(eval_df.head())
display(eval_df[["threshold", "background_excess_threshold", "contrast_z_excess", "dilation_radius", "candidate_area_px", "gt_area_px"]].describe())

## 05-4. Dilation radius 안정성 확인

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for pair, part in eval_df.groupby("color_pair"):
    axes[0].scatter(part["dilation_radius"], part["threshold"], s=14, alpha=0.55, label=pair)
axes[0].set_xlabel("random dilation radius")
axes[0].set_ylabel("contrast threshold")
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].scatter(eval_df["nominal_rgb_distance"], eval_df["threshold"], s=14, alpha=0.55)
axes[1].set_xlabel("generated nominal_rgb_distance")
axes[1].set_ylabel("estimated contrast threshold")
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig(RUNS_ROOT / "dilated_mask_threshold_stability.png", dpi=150)
plt.show()

corr_cols = ["threshold", "background_excess_threshold", "contrast_z_excess", "dilation_radius", "nominal_rgb_distance", "alpha", "color_pull", "width_px"]
display(eval_df[corr_cols].corr(numeric_only=True).round(3))

## 05-5. 함수 사용 형태

In [ ]:
def scratch_threshold_api(raw_image, segformer_or_dilated_mask):
    return estimate_scratch_contrast_threshold(
        image=raw_image,
        candidate_mask=segformer_or_dilated_mask,
        metric="rgb_euclidean",
        mask_quantile=0.90,
        background_quantile=0.90,
        ring_gap=2,
        ring_radius=14,
        aggregation="max",
    )

api_result = scratch_threshold_api(image, candidate_mask)
api_result